In [ ]:
import sys
sys.path.append('../../../code/libs/')

%load_ext autoreload
%autoreload 2
import utils
import viz
import ios
import constants
import text as txtlib

In [2]:
# import os
import pandas as pd
import numpy as np

In [3]:
ROOT = 'abortions_2015_2022/'
metric = 'abortions'
COLS = {'Area':'state_name',
        'Total by location of service': f"{metric}_by_location"}

In [4]:
# state_name	po_total	po_male	
ios.get_files_from_pattern(ios.path_join(ROOT,"Abortions*.csv"))

['abortions_2015_2022/Abortions-Distributed-by-Area-2018.csv',
 'abortions_2015_2022/Abortions-Distributed-by-Area-2019.csv',
 'abortions_2015_2022/Abortions-Distributed-by-Area-2020.csv',
 'abortions_2015_2022/Abortions-Distributed-by-Area-2021.csv',
 'abortions_2015_2022/Abortions-Distributed-by-Area-2015.csv',
 'abortions_2015_2022/Abortions-Distributed-by-Area-2016.csv',
 'abortions_2015_2022/Abortions-Distributed-by-Area-2022.csv',
 'abortions_2015_2022/Abortions-Distributed-by-Area-2017.csv']

In [5]:
files = ios.get_files_from_pattern(ios.path_join(ROOT,"Abortions*.csv"))
data = pd.DataFrame()

for fn in files:
    year = int(fn.split('-')[-1].split('.csv')[0])
    print(year)
    
    tmp = ios.read_csv(fn, index_col=None, skiprows=[0], usecols=['Area','Total by location of service'])
    
    column = f"{metric}_by_residence"
    tmp_by_residence = ios.read_csv(fn, index_col=None, skiprows=[c for c in range(0,54) if c!= 1]).loc[[0],:].set_index('Area').drop(columns=[str(year)]).T.reset_index()
    tmp_by_residence.rename(columns={'index':'state_name', 'Total by residence':column}, inplace=True)
    tmp_by_residence.loc[:,'state_name'] = tmp_by_residence.loc[:,'state_name'].apply(lambda v: str(v).replace('*','').replace('\n',' ').split('(')[0].strip())
    tmp_by_residence.drop([51,52,53,54,55,56,57], inplace=True)
    tmp_by_residence.loc[:,'year'] = year
    tmp_by_residence.set_index(['state_name','year'], inplace=True)
    tmp_by_residence.loc[:,column] = tmp_by_residence.loc[:,column].apply(lambda v: str(v).replace(',',''))
    tmp_by_residence.columns.name = None
    
    cols = COLS.copy()
    
    tmp = tmp[sorted(cols.keys())]
    tmp.rename(columns=cols, inplace=True)
    tmp.loc[:,'year'] = year
    tmp.loc[:,'state_name'] = tmp.loc[:,'state_name'].apply(lambda v: str(v).replace('*',''))
    
    
    for c in tmp.columns:
        if c.startswith(metric):
            print(c)
            tmp.loc[:,c] = tmp.loc[:,c].apply(lambda v: str(v).replace(',',''))
            tmp.loc[:,c] = tmp.loc[:,c].apply(lambda v: None if v in ['--', '', ' ', '-- ', ' -- '] else v)
            tmp.loc[:,c] = tmp.loc[:,c].astype(float)
            tmp.drop([52,53,54], inplace=True)
    
    tmp.set_index(['state_name','year'], inplace=True)
    ny = 'New York State' if tmp.query("state_name=='New York'").shape[0] == 0 else 'New York'
    tmp.loc[ny, 'abortions_by_location'] = tmp.loc[ny, 'abortions_by_location'].iloc[0] + tmp.loc["New York City",'abortions_by_location'].iloc[0]
    tmp.drop('New York City', inplace=True)
    tmp.rename(columns={ny:'New York'}, inplace=True)
    
    tmp2 = tmp.join(tmp_by_residence)

    print(tmp_by_residence.shape, tmp.shape, tmp2.shape)
    data = pd.concat([data, tmp2], ignore_index=False)

2018
abortions_by_location
(51, 1) (51, 1) (51, 2)
2019
abortions_by_location
(51, 1) (51, 1) (51, 2)
2020
abortions_by_location
(51, 1) (51, 1) (51, 2)
2021
abortions_by_location
(51, 1) (51, 1) (51, 2)
2015
abortions_by_location
(51, 1) (51, 1) (51, 2)
2016
abortions_by_location
(51, 1) (51, 1) (51, 2)
2022
abortions_by_location
(51, 1) (51, 1) (51, 2)
2017
abortions_by_location
(51, 1) (51, 1) (51, 2)


In [6]:
data

,,abortions_by_location,abortions_by_residence
state_name,year,,
Alabama,2018,6484.0,7928
Alaska,2018,1283.0,1373
Arizona,2018,12438.0,12588
Arkansas,2018,3069.0,3551
California,2018,NaN,347.0
...,...,...,...
Virginia,2017,15381.0,16235
Washington,2017,17207.0,17116
West Virginia,2017,1436.0,1743


In [7]:
ios.save_csv(data, 'abortions_2015_2022.csv')